# PTM3D notebook 03: an asteroid on a Keplerian orbit

This notebook uses a body that is not one of the built-in planets or moons. It is placed on a
two-body orbit computed from its orbital elements, with a spin state you choose, and its shape is
a sphere of triangular faces generated by the model. No planetary ephemeris or shape file is
needed. The only external file is the leapseconds kernel (about 5 kB), because SPICE converts the
UTC `start_time`.

The notebook is about thermal inertia, $\Gamma = \sqrt{k \rho c_p}$, the property that asteroid
radiometry measures through how far the surface temperature lags a rapid cycle of sunlight. Two
runs at $\Gamma = 100$ and 200 SI, identical otherwise, show the signature. Section 4 uses them to
show which configuration keys a body defined this way uses, the opposite of the built-in bodies of
notebooks 01 and 02.

**Cost:** three runs of about 20 seconds each on four cores, on a 1,280-face sphere, with two
rotations of output after 50 rotations of spin-up.

**See also:** [notebook 01](01_moon_diurnal.ipynb) for a built-in body and for the results reader
used here, and [notebook 04](04_asteroid_spice.ipynb) for a real ephemeris and shape model.

In [1]:
from ptm import paths, run as engine

CONFIG = "examples/quickstart_asteroid_diurnal.yaml"
CHECK = "examples/quickstart_asteroid_diurnal_ti200_check.yaml"
pre = paths.preflight([CONFIG, CHECK, "data/kernels/lsk/naif0012.tls"],
                      engine="ptm_thermal_sim")
print(pre)
pre.raise_if_missing()

found    <repo>/build/bin/ptm_thermal_sim
found    examples/quickstart_asteroid_diurnal.yaml
found    examples/quickstart_asteroid_diurnal_ti200_check.yaml
found    data/kernels/lsk/naif0012.tls

Repo root: <repo>
Engine:    <repo>/build/bin/ptm_thermal_sim


<Report 4/4 found>

## 1. Run it

The run takes about 20 seconds. `body_radius_km` tells the reader the length unit of `mesh.ply`,
which the file does not record.

In [2]:
run = engine.run_engine(pre.root / CONFIG, echo=True)
base = run.results(body_radius_km=5.0)

Running: <repo>/build/bin/ptm_thermal_sim --config <repo>/examples/quickstart_asteroid_diurnal.yaml
(cwd=<repo>)

╔═══════════════════════════════════════════════════════════════╗
║    *      .    *     \   ||    //      * .   .      *         ║
║       .      *     \  \  ||   //    //       .             .  ║
║  *      .       \   \  \  ☀  //    //      .      *           ║
║     .         *  \   ...-'''''-...//    *         .           ║
║           _ ... .-'''''   PTM3D     ``''-.....,__      .      ║
║   __,.-''    Planetary Thermophysical Model 3D     `'-..., _  ║
║  ═══════════════════════════════════════════════════════════  ║
║      Coupled Thermal-Illumination for Planetary Bodies        ║
╚═══════════════════════════════════════════════════════════════╝
Successfully loaded configuration from: <repo>/examples/quickstart_asteroid_diurnal.yaml
✓ Configuration loaded from: <repo>/examples/quickstart_asteroid_diurnal.yaml
Note: body 'Quickstart Asteroid' is not one of the implemen

Using Keplerian orbit propagation for Quickstart Asteroid
[eq-seed] spin-up seed: seed_window=auto (default), seed_seasonal=off (default) -> off. Pin both in the config to reproduce a specific earlier output.
========== Thermal Equilibration Configuration ==========
Body: Quickstart Asteroid
Synodic day (solar day): 0.20836 Earth days
Equilibration (configured): 50.00 day/night cycles  (thermal.equilibration_orbits)
Equilibration (integrated): 50.00 day/night cycles = 1 pass(es) x 50.00-cycle forcing block  [default]
Seed surface anchor: lumped  (cycle-mean lumped-substrate anchor; the run's per-face census and its QUADRATURE are printed by the solver)
Spin-up seed: physical  (shared per-face conductive steady profile, identical on every solver path)
Equilibration (analytic seed, CPU paths only): 0 day/night cycles  (OFF -- spin-up is the configured equilibration under the REAL forcing)
Total equilibration period: 10.42 Earth days
Output window (for export): 0.42 Earth days
Requested i

Calculating illumination:  [>                                                 ] 0% | elapsed 0s


Calculating illumination:  [============>                                     ] 25% | elapsed 0s, ETA 1s


Calculating illumination:  [=========================>                        ] 50% | elapsed 1s, ETA 1s


Calculating illumination:  [=====================================>            ] 75% | elapsed 1s, ETA 0s


... (4 outputs left out)

=== Equilibration spin-up estimate (report-only; nothing is changed) ===
Target accuracy      : 0.1 K   (thermal.equilibration_target_K)
Spin-up this run does: 50 cycles   (analytic seed + thermal.equilibration_orbits)
Model: n_req = (tau1/P) * ln(E0/target), with tau1 the slowest
eigenvalue of the ACTUAL discrete column (Robin top, Neumann
bottom, 3 inverse iterations) and E0 the seed's departure from the
conductive steady profile ANCHORED AT ITS OWN SURFACE. That anchor
is an assumption: it prices growing the interior gradient, not
reaching an unknown final surface temperature.
Reported per population -- the all-face median hides the lit
faces behind a frozen dark majority (measured: median drift over
50->100 lunations is 0.0 K all-face but -6.12 K on lit faces).
CALIBRATION, so you know what n_req is worth: this is a SINGLE-MODE
model -- the seed's error is assumed to decay as exp(-t/tau1).
Checked once against a measured spin-up, on the production 50-layer
lunar grid: a lit column 

Progress 55/192


Progress 109/192


Progress 163/192
Progress 192/192
Export complete: 192/192
Binary export completed successfully!
Data successfully written in binary format to: output/quickstart_asteroid_diurnal
[Statistics] Exporting statistics over integration window...
[exportStatisticsBinary] Computing and exporting statistics...
Face count: 1280
Time steps: 192
Computing illumination statistics...
Computing direct solar statistics...
Computing primary reflected statistics...
Computing primary IR statistics...
Computing temperature statistics...
[exportStatisticsBinary] Statistics export completed successfully!
[Statistics] Exported to output/quickstart_asteroid_diurnal/statistics
[albedo] effective surface albedo (flux-weighted over the DIRECT beam, the value the BC absorbed with): global A_eff = 0.05 over 1226/1280 ever-lit faces; per-face -> output/quickstart_asteroid_diurnal/statistics/effective_albedo.bin (uint32 n + float32[1280], NaN where never lit).
Reduced over the INTEGRATION window, abs steps [2400,259

## 2. The first look

One output step from the second, settled rotation.

In [3]:
fig = base.plot_globes("temperature", step=120)

The map shows a warm afternoon side, a cold night side, and a strong north/south
asymmetry that needs to be understood before anything else, because it sets which faces the
rest of the notebook may use.

In [4]:
import numpy as np

subsolar = np.array([base.subsolar_latlon(s) for s in base.steps("temperature")])
print(f"sub-solar latitude : {subsolar[:, 0].min():+.2f} to {subsolar[:, 0].max():+.2f} deg")
print(f"sub-solar longitude: {subsolar[:, 1].min():+.0f} to {subsolar[:, 1].max():+.0f} deg "
      f"over {len(subsolar)} steps")

sub-solar latitude : -23.02 to -23.00 deg
sub-solar longitude: -178 to +178 deg over 192 steps


The subsolar latitude stays near -23° for the whole run, and only the longitude changes. So the
only variation on this timescale is the diurnal cycle, and the two poles differ. Poleward of 67°N
the Sun never rises, and poleward of 67°S it never sets, which explains the asymmetry of the
globes.

The latitude follows from the configuration. The spin pole is given in the equatorial frame
(`pole_dec_deg: 90`), and the orbit is inclined to that frame, so the effective obliquity is about
23°. Nothing below depends on its value, only on its being constant over the run. The polar faces
are in permanent sunlight or permanent shadow, so the comparisons below use a band of faces near
the equator rather than extremes over the whole body.

## 3. The configuration

[`examples/quickstart_asteroid_diurnal.yaml`](../../../examples/quickstart_asteroid_diurnal.yaml)

In [5]:
print((pre.root / CONFIG).read_text())

# PTM3D quickstart: a generic-body (Keplerian) asteroid diurnal run on a small
# icosphere (ico3, 1280 faces).
#
# Laptop-friendly (CPU-only, <1 GB RAM; ~23 s measured on an 8-core machine
# with thermal.threads=4, well under a minute of wall time).
# SELF-CONTAINED: unlike the Moon/Horizons quickstarts, this run needs NO
# external data at all — no SPICE kernels, no DEM, no albedo/property maps.
# The body is fictional (there is no real "Quickstart Asteroid"); its orbit is
# propagated analytically (Keplerian two-body) and its shape is a synthetic
# icosphere, so nothing outside this file is fetched or required.
#
# All paths are relative — run FROM THE REPOSITORY ROOT:
#
#     ./build/bin/ptm_thermal_sim --config examples/quickstart_asteroid_diurnal.yaml
#
# Outputs land in output/quickstart_asteroid_diurnal/ (per-face statistics +
# per-timestep binaries). See docs/tutorials/notebooks/03_asteroid_thermal.ipynb
# for a full walkthrough of the outputs.
#
# ----------------------------

The body is a sphere of 1,280 faces with a radius of 5 km, on a circular orbit at $a = 2.7$ AU in
the middle of the main belt. It is not a real object. The solar flux there is
$1361/2.7^2 \approx 187$ W/m², 14% of the flux at Earth. Such bodies are cold and, being dark,
comparatively bright in the thermal infrared, which is why their surface properties are measured
by infrared radiometry rather than visible photometry. The rotation period is 5 hours.

The configuration sets `ks = kd = 0.008333` W/m/K, `rhos = rhod = 1500` kg/m³, `cp0 = 800` J/kg/K
and `chi = 0`, so

$$\Gamma = \sqrt{0.008333 \times 1500 \times 800} = 100.0\ \mathrm{SI},$$

the same at every depth and temperature. Equal surface and deep values make the regolith a single
uniform layer, so one number describes it. 100 SI is in the range of regolith. Loose regolith like
the Moon's is near 50 SI, and bare rock and boulders 400 to 2500 SI. Values far outside about 32 to
316 SI can reach the limits of the numerical grid, because a very low thermal inertia gives a
skin depth below a millimeter and a very high one needs a long spin-up.

For a body defined this way, plain property names (`ks`, `kd`, `rhos`, `cp0`, `Qb`, `chi`,
`albedo`, ...) are used only in `generic_body.surface:`, not in a top-level `thermophysical:`
block ([`docs/KNOWN_LIMITATIONS.md`](../../KNOWN_LIMITATIONS.md)). This configuration therefore has
only `use_default_albedo: false` at the top level.

## 4. Changing a property of a body defined in the configuration

This is the reverse of notebook 01. On a built-in body the plain property names are not used, and
a change has to go through `thermophysical.*_override`, `constant_thermal_inertia`, a map or a
command-line option. Here every key in `generic_body.surface:` is used: `albedo`, `emissivity`,
`ks`, `kd`, `rhos`, `rhod`, `cp0`, `H`, `Qb` and `chi`. Changing any of them changes the result.
This is a reason to define a body this way even when a built-in body would almost do.

Where the key is placed is still important. The same names in a top-level `thermophysical:` block are not
used. `thermophysical.ks` next to `generic_body.surface.ks` leaves the run unchanged, and the log
lists such keys in a line beginning `[config] WARNING:`. Notebook 01, section 10 shows
`ptm.engine_config.unknown_keys` finding the same kind of mistake before a run.

The `*_override` keys also work for these bodies, and they are the only keys whose value the log
confirms. The cells below use both.

In [6]:
banner = run.lines.index(next(l for l in run.lines if l.startswith("Using generic body")))
print("\n".join(run.lines[banner:banner + 10]))

Using generic body: Quickstart Asteroid
Semi-major axis: 2.7 AU
Eccentricity: 0
Spin period: 5 hours
Orbit epoch: JD 2451545 TDB  (mean anomaly 0 deg at this epoch)
Spin epoch:  JD 2451545 TDB  (prime meridian W0 = 0 deg at this epoch)
Solar constant: 186.694 W/m^2
Synodic day: 5.00064 hours
Albedo: 0.05
Emissivity: 0.95


The log confirms the body name, orbit, spin period, solar flux, albedo, emissivity, radius and
ephemeris, and does not mention `ks`, `kd`, `rhos`, `rhod`, `cp0`, `Qb` or `chi`. A change in
`generic_body.surface` has no log line of its own, so confirm it with the two checks of notebook 01.

1. Two runs must differ. A checksum of `statistics/temperature_max.bin` distinguishes a real change
   from one that had no effect.
2. The change must have the right sign. A higher thermal inertia must reduce the diurnal range.

`examples/quickstart_asteroid_diurnal_ti200_check.yaml` is this configuration with `ks` and `kd`
multiplied by four, so $\Gamma = 200$ SI. The third run makes the same change through
`thermophysical.ks_override`, which the log confirms.

In [7]:
import hashlib

from ptm import Sweep

override = Sweep(pre.root / CONFIG,
                 grid={"thermophysical.ks_override": [0.033333333]},
                 overrides={"thermophysical.kd_override": 0.033333333},
                 outdir=pre.root / "output/asteroid_ks_override",
                 results_kwargs={"body_radius_km": 5.0})
override.run()
check = engine.run_engine(pre.root / CHECK, show=("Simulation completed in",)
                          ).results(body_radius_km=5.0)

band = base.faces_in_band(-10.0, 10.0)
print(f"\nequatorial band: {band.size} of {base.n_faces} faces\n")
for tag, res in (("100 SI, as shipped        ", base),
                 ("200 SI, generic_body.surface", check),
                 ("200 SI, thermophysical.*_override", override.points[0].results)):
    hi = res.stat("temperature", "max")[band].mean()
    lo = res.stat("temperature", "min")[band].mean()
    digest = hashlib.md5(
        (res.path / "statistics" / "temperature_max.bin").read_bytes()).hexdigest()
    print(f"  {tag:34s} Tmax {hi:6.1f} K   Tmin {lo:6.1f} K   "
          f"amplitude {hi - lo:5.1f} K   {digest[:8]}")
print("\n" + next(l for l in (override.points[0].dir / "engine.log").read_text().splitlines()
                  if "applied at solver:" in l))

[sweep] engine <repo>/build/bin/ptm_thermal_sim (md5 886d9b49b1083d2984b35328264035da)
[sweep] axis thermophysical.ks_override: 1 values -> config key thermophysical.ks_override


[sweep] ok   ks_override-0p0333333 (7.8 s)
[sweep] 1 complete, 0 failed, 0 skipped, 7.8 s


    Simulation completed in 9.662 seconds (9662 ms)

equatorial band: 220 of 1280 faces

  100 SI, as shipped                 Tmax  207.9 K   Tmin  147.0 K   amplitude  60.9 K   120ba9b1
  200 SI, generic_body.surface       Tmax  196.9 K   Tmin  158.6 K   amplitude  38.3 K   c28539f6
  200 SI, thermophysical.*_override  Tmax  196.9 K   Tmin  158.6 K   amplitude  38.3 K   0cc08625

[Thermo override] applied at solver: ks 0.033333 -> 0.0333333 W/m/K; kd 0.033333 -> 0.0333333 W/m/K


The mean maximum at the equator fell and the mean minimum rose, so the range narrowed, which is
the expected sign. The checksums differ, so the change reached the thermal model.

The two runs at 200 SI agree in every printed digit, and their maximum temperatures differ by at
most $10^{-4}$ K on any face. `generic_body.surface.ks` and `thermophysical.ks_override` are two
ways to make the same change. Use the surface block when the value is part of the body's
definition, and the override when it belongs to the experiment, for example in a sweep or to leave
the example configuration unchanged. Only the override is confirmed in the log, so a sweep over
the surface block depends on the checksum test.

## 5. The signature of thermal inertia

The curve below shows temperature against local solar time, the hour angle of the Sun on a
24-hour clock with local noon at 12 h, whatever the rotation period. For each face near the
equator, local noon is the step at which its `direct_solar` flux peaks, and its temperatures are
placed on the clock relative to that noon. `local_solar_time_curve` does this and averages over
the band of faces, which smooths out differences between individual faces. You choose which steps
to use, here the second of the two rotations written.

In [8]:
from ptm import plots

plt = plots.styled()
curves = {}
for tag, res in (("100 SI", base), ("200 SI", check)):
    steps = res.steps("temperature")
    curves[tag] = res.local_solar_time_curve(steps=steps[len(steps) // 2:])

fig, ax = plt.subplots(figsize=(7, 4.0))
for (tag, (lst, T, lo, hi, n_faces)), color in zip(curves.items(), ("#2a78d6", "#eb6834")):
    ax.plot(lst, T, color=color, lw=2, label=rf"$\Gamma$ = {tag}")
    ax.fill_between(lst, lo, hi, color=color, alpha=0.12, linewidth=0)
    print(f"{tag}: peak at local solar time {lst[np.argmax(T)]:.2f} h, "
          f"amplitude {T.max() - T.min():.1f} K")
ax.axvline(12, color="#898781", lw=1, ls="--")
ax.set_xlabel("Local solar time (h)")
ax.set_ylabel("Surface temperature (K)")
ax.set_title("Thermal-inertia signature: amplitude damping and phase lag")
ax.set_xlim(0, 24)
ax.set_xticks(range(0, 25, 3))
ax.legend(frameon=False)
ax.grid(True, linewidth=0.5)
fig.tight_layout()

100 SI: peak at local solar time 13.25 h, amplitude 60.6 K
200 SI: peak at local solar time 13.75 h, amplitude 38.1 K


Both effects appear. Doubling the thermal inertia delays the peak by about half an hour and
reduces the diurnal range by nearly 40% (Spencer et al., 1989). The shaded bands show the 25th to
75th percentile range between faces at the same local time. They are narrow, so the average over
the band is representative.

A larger delay does not mean a larger difference between morning and afternoon. The difference
$T(\text{noon}+x) - T(\text{noon}-x)$ is positive in both runs, with the afternoon warmer than the
corresponding morning, but it is smaller at 200 SI, because over this range the amplitude falls
faster than the delay grows. Thermophysical fits therefore use the whole curve, amplitude and phase
together, to determine $\Gamma$.

## 6. Interpretation

Thermal inertia, $\Gamma = \sqrt{k \rho c_p}$, controls how far the diurnal thermal wave penetrates
before the next sunset reverses it. The penetration is the skin depth
$\ell_s = \sqrt{\kappa P/\pi}$, about 6.3 mm for this configuration and its 5-hour rotation. The
shape of the whole curve is governed by the thermal parameter

$$\Theta = \frac{\Gamma \sqrt{\omega}}{\varepsilon \sigma T_{eq}^3}$$

(Spencer, Lebofsky & Sykes, 1989), with $\omega$ the rotation rate and $T_{eq}$ the subsolar
equilibrium temperature for zero thermal inertia. When $\Theta \to 0$ the surface is in
instantaneous equilibrium, with a large range and no delay. When $\Theta \to \infty$ it is
perfectly conducting, with no diurnal variation. The step from 100 to 200 SI is a short move along
that curve.

The Yarkovsky effect follows from the same physics. Because the peak temperature comes after local
noon, the asteroid emits more thermal radiation from its afternoon side than a body in
instantaneous equilibrium would, and the emitted photons carry momentum. For a prograde rotator
the recoil has a small component along the orbital velocity, which over millions of years moves
the semimajor axis outward, and inward for a retrograde rotator (Bottke et al., 2006). Thermal
inertia sets the size of the effect, which vanishes in both limits and is largest at intermediate
$\Theta$, typical of small near-Earth asteroids. Measuring thermal inertia and modeling Yarkovsky
drift are the same problem in practice.

The procedure here is the one used with real observations. Thermal infrared photometry measures
the flux integrated over the disk at one or more wavelengths and rotational phases. A
thermophysical model with the body's shape, spin and albedo is run for a grid of thermal inertias,
and the value that reproduces the observed amplitude and phase is reported. Thermal infrared
imaging from a flyby or rendezvous measures temperature against local time directly, as in the
figure above.

A thermal lightcurve cannot be shown with this body. The flux integrated over the disk varies as a
nonspherical body turns a changing outline toward the observer, so it depends on the shape as much
as on the thermal properties. Notebook 04 uses a real shape model.

## 7. Exercises

Each exercise needs one or more runs of about 20 seconds, with changes to the configuration and the
two cells above.

1. Thermal inertia. At fixed density and heat capacity $\Gamma \propto \sqrt{k_s}$, so
   $\Gamma =$ 32, 100 and 316 SI need `generic_body.surface.ks` = `kd` = 0.000853, 0.008333 and
   0.0833 W/m/K. Plot the range and the time of the peak against $\Gamma$. Outside this range, expect
   the numerical limits described in section 3.

2. Rotation period. Change `generic_body.spin.period_hours` to 2 and 10 hours at $\Gamma = 100$.
   The curve depends on $\Theta \propto \Gamma\sqrt{\omega}$, so a faster rotator should show a
   narrower range at the same thermal inertia. Scale `thermal.equilibration_steps_per_day` in
   inverse proportion to the period, so the number of samples per rotation stays the same.

3. Eccentricity. Set `generic_body.orbit.eccentricity: 0.2`. The output window here is about 10
   hours, far too short for the distance to the Sun to change over a 4.4-year orbit, so extend
   `simulation.duration_days` to a large fraction of the orbital period, which takes much longer.
   Look for a slow drift of the mean temperature under the same diurnal cycle.